# Modul 11 · Zeitreihen und Forecasting
**Data Science 1** · linkit courses

Was passiert morgen, und wann läuft etwas aus dem Ruder? Das Skript rechnet am Strom, du rechnest am Gas der Nordwerk GmbH. Du findest die Ausfalltage mit Nachbuchung, hängst den Plan an, zerlegst die Reihe, rechnest Autokorrelation und einfache Glättung von Hand, baust Messlatten mit dem richtigen Horizont und eine Regression mit Heizgradtagen ohne Leckage. Du prüfst sie mit rollierendem Ursprung und mit Intervallen und überwachst am Ende die Reste des Stroms an freien Tagen mit CUSUM, EWMA und Shewhart-Karte.

**Lernziele**
1. Zeitreihen vorbereiten: Zeitindex, Lücken, Kalender und Zerlegung
2. Einfache Prognosen als Messlatte bauen und mit passenden Fehlermaßen bewerten
3. Exponentielle Glättung anwenden und ihre Grenzen bei Kalendereffekten erkennen
4. Regression mit Plan, Kalender und Wetter ohne Leckage aufbauen
5. Prognosen mit rollierendem Ursprung fair vergleichen
6. Prognoseintervalle bilden und ihre Überdeckung prüfen
7. Mit Regelkarten, CUSUM und EWMA Abweichungen früh erkennen

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus. Der erste Import lädt im Browser SciPy, statsmodels und scikit-learn, das dauert einmalig etwa 15 Sekunden.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis zehn Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Wo ein späteres TODO ein früheres Ergebnis braucht, springt eine Vorgabe-Zelle ein, solange das frühere offen ist. Du kannst die TODOs also auch einzeln bearbeiten.
- Mindestziel sind TODO 1 bis 13. Die beiden Bonusaufgaben vertiefen.
- In diesem Modul gibt es keinen Zufall: Regression, Quantile, Glättung und Karten liefern bei jedem Lauf dasselbe.
- Gib Ergebnisse mit `float(...)`, `round(...)` oder der Hilfsfunktion `zeige` aus, dann steht im Browser und in Colab dasselbe da. Jede Grafikzelle endet mit `plt.show()`.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["energie.csv", "produktion.csv", "sensordaten.csv", "maschinen.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Zeitreihen lesen
*Lerneinheit 1 · Skript Abschnitt 1*

Eine Zeitreihe braucht einen Datumsindex mit Frequenz: `asfreq("D")` legt einen Eintrag je Kalendertag an, und viele Verfahren in statsmodels verlangen diese Frequenz. Fällt die Datenübertragung aus, fehlt ein Tag, und der Folgetag enthält den Verbrauch beider Tage. Den Kalender mit Feiertagen und Werksruhe liefert der Plan aus `produktion.csv`. Eine Zerlegung trennt Trend, Saison und Rest, die Autokorrelation zeigt, wie stark ein Wert mit dem Wert $k$ Tage davor zusammenhängt.

Die nächste Zelle lädt `energie.csv` wie im Skript. `gas_roh` hält die Rohwerte des Gases fest, sie bleiben gleich, auch wenn du `energie` bereinigst. `TRAIN_ENDE` beendet das Trainingsjahr, `pruef` markiert die 365 Tage danach. Die Hilfsfunktion `zeige` druckt Zahlen mit 6 gültigen Stellen, Zeitstempel als Datum, Tabellen mit ihrer Größe und lange Arrays mit Länge und Summe.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing
from sklearn.linear_model import LinearRegression

energie = pd.read_csv("daten/energie.csv", parse_dates=["datum"], index_col="datum").asfreq("D")
gas_roh = energie["gas_kwh"].copy()
TRAIN_ENDE = "2025-09-30"
pruef = energie.index > TRAIN_ENDE
print(energie.shape, energie.index.freqstr, int(pruef.sum()), energie.isna().sum().to_dict())


def zeige(*werte):
    """Druckt Zahlen mit 6 gültigen Stellen, damit im Browser und in Colab dieselben Ziffern stehen.
    Zeitstempel erscheinen als Datum, Tabellen mit ihrer Größe, lange Arrays mit Länge und Summe. None bleibt None."""
    def text(w):
        if w is None or isinstance(w, (str, bool, np.bool_)):
            return str(w)
        if isinstance(w, pd.Timestamp):
            return w.strftime("%d.%m.%Y")
        if isinstance(w, dict):
            return "{" + ", ".join(f"{text(k)}: {text(v)}" for k, v in w.items()) + "}"
        if isinstance(w, tuple):
            return "(" + ", ".join(text(v) for v in w) + ")"
        if isinstance(w, pd.DataFrame):
            return f"Tabelle {w.shape[0]} × {w.shape[1]}"
        if np.ndim(w) == 0:
            return f"{float(w):.6g}"
        w = list(np.ravel(np.asarray(w, dtype=object)))
        if len(w) > 12:
            try:
                return f"[{len(w)} Werte, Summe {float(np.nansum(np.asarray(w, dtype=float))):.6g}]"
            except (TypeError, ValueError):
                return f"[{len(w)} Werte]"
        return "[" + " ".join(text(v) for v in w) + "]"
    print(*[text(w) for w in werte])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(energie.index, energie["gas_kwh"] / 1000, lw=0.7, color="tab:red")
ax.set_ylabel("Gas in MWh je Tag", color="tab:red")
ax2 = ax.twinx()
ax2.plot(energie.index, energie["aussentemp_c"], lw=0.7, color="tab:blue", alpha=0.6)
ax2.set_ylabel("Außentemperatur in °C", color="tab:blue")
ax.axvline(pd.Timestamp(TRAIN_ENDE), color="gray", ls="--", lw=1)
ax.set_title("Gas folgt der Temperatur, Trainingsjahr links der Linie")
plt.show()

### TODO 1 · Ausfälle und Nachbuchung beim Gas
Im Skript fehlen an drei Tagen Strom und Gas, und der Folgetag trägt den Verbrauch beider Tage. Hier siehst du dir das beim Gas an. Rechne in 1a und 1b mit `gas_roh`, dann liefert die Zelle auch beim zweiten Ausführen dasselbe.
1. `luecke`: True an den Tagen, an denen `gas_roh` fehlt. `nachbuchung`: True am Tag danach.
2. `paare`: eine Tabelle mit einer Zeile je Ausfalltag und den Spalten `vortag` und `nachbuchung`, dem Gas am Tag vor dem Ausfall und am Tag danach. `feb_mit`: das mittlere Gas im Februar 2026 aus `gas_roh`.
3. Setze in `energie` an den Nachbuchungstagen Strom und Gas auf fehlend, wie im Skript. `feb_ohne`: das Februarmittel danach aus `energie["gas_kwh"]`.

In [ ]:
# TODO 1a
luecke = None
nachbuchung = None

# TODO 1b
paare = None
feb_mit = None

# TODO 1c
feb_ohne = None

zeige(None if luecke is None else list(energie.index[luecke]), None if paare is None else paare.to_numpy(), feb_mit, feb_ohne, int(energie["gas_kwh"].isna().sum()))

In [ ]:
check("TODO 1a Ausfalltage", lambda: int(luecke.sum()) == 3 and int(nachbuchung.sum()) == 3
      and [t.strftime("%d.%m.%Y") for t in energie.index[luecke]] == ["12.03.2025", "19.11.2025", "04.02.2026"]
      and [t.strftime("%d.%m.%Y") for t in energie.index[nachbuchung]] == ["13.03.2025", "20.11.2025", "05.02.2026"])
check("TODO 1b Nachbuchung gegen Vortag", lambda: paare["vortag"].tolist() == [12715, 9858, 17714] and paare["nachbuchung"].tolist() == [17802, 19717, 34854]
      and abs(feb_mit - 13561.4) < 0.1)
check("TODO 1c beide Tage fehlend", lambda: int(energie["gas_kwh"].isna().sum()) == 6 and int(energie["strom_kwh"].isna().sum()) == 6
      and bool(energie.loc[nachbuchung, "gas_kwh"].isna().all()) and abs(feb_ohne - 12742.5) < 0.1)

In [ ]:
# Vorgabe: Ist TODO 1 noch offen, setzt diese Zelle die Nachbuchungen auf fehlend. In der Lösung ändert sie nichts.
if nachbuchung is None:
    energie.loc[gas_roh.isna().shift(1, fill_value=False), ["strom_kwh", "gas_kwh"]] = np.nan
strom = energie["strom_kwh"]
gas = energie["gas_kwh"]
print(int(strom.isna().sum()), int(gas.isna().sum()))

Gas fehlt am 12. März 2025, am 19. November 2025 und am 4. Februar 2026. Die Nachbuchungen liegen bei 17.802, 19.717 und 34.854 kWh, am Vortag des Ausfalls bei 12.715, 9.858 und 17.714. Im Februar 2026 ist der doppelte Tag ein kräftiger Ausreißer: Mit ihm liegt das Monatsmittel bei 13.561,4 kWh, ohne ihn bei 12.742,5. Ab hier heißen die bereinigten Reihen `strom` und `gas`, mit je 6 fehlenden Tagen.

### TODO 2 · Plan anhängen, Gas nach Tagesart
1. `produktion`: `produktion.csv` mit `datum` als Zeitstempel. `plan`: je Tag die Zahl der Schichten (`schichten`, verschiedene Werte in `schicht`) und die Summe von `soll_stueck` (`soll`), wie im Skript.
2. Hänge `schichten` und `soll` an `energie` an, Tage ohne Zeile im Plan mit 0. Mit `energie[["schichten", "soll"]] = ...` kannst du die Zelle mehrmals ausführen. Ergänze die Spalten `wochentag` (0 = Montag) und `arbeitstag` (mindestens eine Schicht).
3. `jahreszeit`: eine Series mit „Winter“ für Dezember bis Februar, „Sommer“ für Juni bis August und „Übergang“ sonst. `gas_art`: das mittlere Gas im Trainingsjahr je Jahreszeit (Zeilen in der Reihenfolge Winter, Übergang, Sommer) und Schichtzahl (Spalten 0, 1, 3). `pruef_art`: die Zahl der Tage im Prüfjahr je Schichtzahl als Series.

In [ ]:
# TODO 2a
produktion = None
plan = None

# TODO 2b: Spalten schichten, soll, wochentag und arbeitstag in energie

# TODO 2c
jahreszeit = None
gas_art = None
pruef_art = None

print(None if gas_art is None else gas_art.round(0))
zeige(None if pruef_art is None else pruef_art.to_dict())

In [ ]:
check("TODO 2a Plan", lambda: list(plan.columns) == ["schichten", "soll"] and len(plan) == 505 and set(plan["schichten"]) == {1, 3} and int(plan["soll"].sum()) == int(produktion["soll_stueck"].sum()))
check("TODO 2b Spalten in energie", lambda: energie["schichten"].value_counts().sort_index().to_dict() == {0: 225, 1: 34, 3: 471} and int(energie["soll"].sum()) == int(plan["soll"].sum())
      and energie["arbeitstag"].dtype == bool and int(energie["arbeitstag"].sum()) == 505 and energie.loc["2025-10-03", "wochentag"] == 4)
check("TODO 2c Gas je Jahreszeit und Schichtzahl", lambda: list(gas_art.index) == ["Winter", "Übergang", "Sommer"] and list(gas_art.columns) == [0, 1, 3]
      and np.allclose(gas_art.loc["Winter"], [6460.9, 11403.0, 15941.3], atol=1) and np.allclose(gas_art.loc["Übergang"], [1299.3, 6414.1, 8211.0], atol=1)
      and abs(gas_art.loc["Sommer", 0] - 301.0) < 1 and abs(gas_art.loc["Sommer", 3] - 4671.4) < 1 and np.isnan(gas_art.loc["Sommer", 1])
      and pruef_art.to_dict() == {0: 107, 1: 22, 3: 236})

In [ ]:
# Vorgabe: Ist TODO 2 noch offen, hängt diese Zelle den Plan an. In der Lösung ändert sie nichts.
if "arbeitstag" not in energie:
    _plan = pd.read_csv("daten/produktion.csv", parse_dates=["datum"]).groupby("datum").agg(schichten=("schicht", "nunique"), soll=("soll_stueck", "sum"))
    energie[["schichten", "soll"]] = _plan.reindex(energie.index, fill_value=0)
    energie["wochentag"] = energie.index.dayofweek
    energie["arbeitstag"] = energie["schichten"] > 0
print(list(energie.columns))

Im Winter braucht das Werk an Tagen ohne Produktion im Mittel 6.461 kWh Gas, mit einer Schicht 11.403 und mit drei Schichten 15.941. In der Übergangszeit sind es 1.299, 6.414 und 8.211, im Sommer an freien Tagen nur 301 und mit drei Schichten 4.671. Samstagsschichten gibt es im Sommer keine. Gas hängt an Schichtzahl und Jahreszeit zugleich, die Heizung läuft an Arbeitstagen stärker. Im Prüfjahr gibt es 107 Tage ohne Produktion, 22 mit einer und 236 mit drei Schichten.

### TODO 3 · Wochensummen und STL für Gas
1. `gas_woche`: die Wochensummen von `gas` von Montag bis Sonntag, eine Woche mit fehlendem Tag bleibt leer. `woche_max` und `woche_min`: das Datum (der Sonntag) der Woche mit der höchsten und der niedrigsten Summe.
2. `gas_voll`: `gas` mit den fehlenden Tagen aus dem Wert eine Woche davor. `stl_gas`: STL mit Periode 7 und `robust=True`, angepasst an `gas_voll`.
3. `anteile_gas`: ein Wörterbuch mit den Schlüsseln „Saison“, „Trend“ und „Rest“ und der Varianz des Teils geteilt durch die Varianz von `gas_voll`.

In [ ]:
# TODO 3a
gas_woche = None
woche_max = None
woche_min = None

# TODO 3b
gas_voll = None
stl_gas = None

# TODO 3c
anteile_gas = None

zeige(None if gas_woche is None else int(gas_woche.notna().sum()), woche_max, woche_min, anteile_gas)

In [ ]:
check("TODO 3a Wochensummen", lambda: len(gas_woche) == 105 and int(gas_woche.notna().sum()) == 100 and woche_max == pd.Timestamp("2026-01-18") and woche_min == pd.Timestamp("2025-08-17")
      and abs(gas_woche.max() - 115315) < 1 and abs(gas_woche.min() - 2045) < 1)
check("TODO 3b STL auf lückenloser Reihe", lambda: int(gas_voll.isna().sum()) == 0 and gas_voll.index.freqstr == "D" and gas_voll.loc["2025-03-12"] == gas.loc["2025-03-05"]
      and np.allclose(stl_gas.trend + stl_gas.seasonal + stl_gas.resid, gas_voll))
check("TODO 3c Varianzanteile", lambda: np.allclose([anteile_gas["Saison"], anteile_gas["Trend"], anteile_gas["Rest"]], [0.302, 0.586, 0.164], atol=0.001))

100 Wochen sind vollständig. Die höchste Wochensumme hat die Woche bis zum 18. Januar 2026 mit 115.315 kWh, die niedrigste die Woche bis zum 17. August 2025 mit 2.045 kWh, in der Werksruhe. Beim Gas trägt die Wochensaison 30,2 % der Varianz, der Trend 58,6 % und der Rest 16,4 %. Beim Strom war es umgekehrt, dort trägt die Wochensaison 83,3 %. Der Trend des Gases ist vor allem der Jahresgang: hoch im Winter, niedrig im Sommer.

### TODO 4 · Autokorrelation von Hand
1. `autokorrelation(x, k)`: eine Funktion, die $r_k$ nach der Formel aus dem Skript berechnet, mit dem Mittel und dem Nenner über alle Werte von `x`.
2. `sommer`: `gas` vom 1. April bis 30. September 2025, dort fehlt kein Tag. `r_hand`: ein Wörterbuch mit den Schlüsseln 1, 2 und 7 und $r_k$ aus deiner Funktion. `r_acf`: dasselbe aus `acf`.

In [ ]:
# TODO 4a
autokorrelation = None

# TODO 4b
sommer = None
r_hand = None
r_acf = None

zeige(r_hand, r_acf)

In [ ]:
check("TODO 4a Formel an der kurzen Reihe", lambda: abs(autokorrelation(np.array([5.0, 7.0, 6.0, 9.0, 8.0, 10.0]), 1) - 0.1) < 1e-12)
check("TODO 4b Gas im Sommerhalbjahr", lambda: len(sommer) == 183 and int(sommer.isna().sum()) == 0 and np.allclose([r_hand[1], r_hand[2], r_hand[7]], [0.459, 0.046, 0.539], atol=0.001)
      and all(abs(r_hand[k] - r_acf[k]) < 1e-9 for k in [1, 2, 7]))

Im Sommerhalbjahr liegt die Autokorrelation des Gases bei Lag 1 bei 0,459, bei Lag 2 bei 0,046 und bei Lag 7 bei 0,539, genau wie `acf`. Ohne Heizung bleibt vor allem das Wochenmuster, wie beim Strom. Über zwei Jahre war das Gas bei allen Lags hoch, weil kalte Tage auf kalte Tage folgen.

## 2 · Messlatten: einfache Prognosen und Fehlermaße
*Lerneinheit 2 · Skript Abschnitt 2*

Am Vormittag des Tages $t$ geht die Prognose für den Zieltag $t+1$ hinaus. Der laufende Tag ist noch nicht abgelesen, der letzte bekannte Verbrauch ist zwei Tage alt: Der Horizont ist zwei. Die saisonal naive Prognose nimmt den gleichen Wochentag der Vorwoche. Der MASE teilt den MAE durch den MAE der saisonal naiven Prognose im Training. Der MAPE teilt durch den tatsächlichen Wert und bricht bei kleinen Werten.

Die Funktion `fehlermasse` rechnet MAE, RMSE, MAPE und MASE wie im Skript. Anders als dort bekommt sie den Nenner des MASE als drittes Argument.

In [ ]:
def fehlermasse(y, p, nenner):
    """MAE, RMSE, MAPE und MASE für tatsächliche Werte y und Prognosen p; nenner ist der MAE der saisonal naiven Prognose im Training."""
    e = y - p
    return pd.Series({"MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean()), "MAPE": 100 * (e / y).abs().mean(), "MASE": e.abs().mean() / nenner})

### TODO 5 · Messlatten für Gas mit dem richtigen Horizont
1. `gas_prognosen`: eine Tabelle mit den Spalten `gestern` (Vortag des Zieltags, im Betrieb nicht verfügbar), `zuletzt_abgelesen` und `saisonal_naiv` für Gas.
2. `MASE_NENNER_GAS`: der mittlere absolute Fehler der saisonal naiven Prognose im Trainingsjahr. `tage`: True an den Tagen im Prüfjahr, an denen Gas und alle drei Prognosen vorliegen.
3. `gas_masse`: eine Tabelle mit einer Zeile je Prognose und den Spalten MAE, RMSE, MAPE und MASE auf `tage`, berechnet mit `fehlermasse`.
4. `mape_art`: ein Wörterbuch mit dem MAPE der saisonal naiven Prognose auf `tage` an Arbeitstagen (Schlüssel „Arbeitstage“) und an freien Tagen von Juni bis August (Schlüssel „frei im Sommer“).

In [ ]:
# TODO 5a
gas_prognosen = None

# TODO 5b
MASE_NENNER_GAS = None
tage = None

# TODO 5c
gas_masse = None

# TODO 5d
mape_art = None

print(None if gas_masse is None else gas_masse.round(3))
zeige(MASE_NENNER_GAS, None if tage is None else int(tage.sum()), mape_art)

In [ ]:
check("TODO 5a drei Prognosen", lambda: list(gas_prognosen.columns) == ["gestern", "zuletzt_abgelesen", "saisonal_naiv"]
      and np.allclose(gas_prognosen["zuletzt_abgelesen"], gas.shift(2), equal_nan=True) and np.allclose(gas_prognosen["saisonal_naiv"], gas.shift(7), equal_nan=True))
check("TODO 5b Nenner und Prüftage", lambda: abs(MASE_NENNER_GAS - 2858.8) < 0.1 and int(tage.sum()) == 353 and not bool(tage[~pruef].any()))
check("TODO 5c Fehlermaße", lambda: np.allclose(gas_masse.loc["saisonal_naiv", ["MAE", "MAPE", "MASE"]], [2583.2, 87.5, 0.904], atol=[0.1, 0.1, 0.001])
      and np.allclose(gas_masse.loc["zuletzt_abgelesen", ["MAE", "MAPE"]], [4346.9, 262.2], atol=0.1) and abs(gas_masse.loc["gestern", "MAE"] - 2915.0) < 0.1)
check("TODO 5d MAPE nach Tagesart", lambda: abs(mape_art["Arbeitstage"] - 37.5) < 0.1 and abs(mape_art["frei im Sommer"] - 277.4) < 0.1)

In [ ]:
# Vorgabe: Ist TODO 5 noch offen, legt diese Zelle die Prognosen, den Nenner, die Prüftage und die Fehlermaße an. In der Lösung ändert sie nichts.
if gas_prognosen is None or tage is None or MASE_NENNER_GAS is None or gas_masse is None:
    gas_prognosen = pd.DataFrame({"gestern": gas.shift(1), "zuletzt_abgelesen": gas.shift(2), "saisonal_naiv": gas.shift(7)})
    MASE_NENNER_GAS = (gas[:TRAIN_ENDE] - gas[:TRAIN_ENDE].shift(7)).abs().mean()
    tage = pruef & gas.notna() & gas_prognosen.notna().all(axis=1)
    gas_masse = pd.DataFrame({n: fehlermasse(gas[tage], gas_prognosen.loc[tage, n], MASE_NENNER_GAS) for n in gas_prognosen}).T
print(int(tage.sum()))

Es bleiben 353 Prüftage, dieselben wie beim Strom. Die saisonal naive Prognose hat einen MAE von 2.583,2 kWh und einen MASE von 0,904. „Zuletzt abgelesen“ liegt mit 4.346,9 weit dahinter, selbst das nicht verfügbare „gestern“ mit 2.915,0. Der MAPE sagt etwas anderes: 87,5 % für die saisonal naive Prognose, 262,2 % für „zuletzt abgelesen“. An Arbeitstagen liegt der MAPE der saisonal naiven Prognose bei 37,5 %, an freien Sommertagen bei 277,4 %. Wenige hundert kWh im Nenner bestimmen den Mittelwert. Beim Gas taugt der MAPE nicht als Hauptmaß.

## 3 · Exponentielle Glättung
*Lerneinheit 3 · Skript Abschnitt 3*

Die einfache exponentielle Glättung mischt den neuen Wert mit der alten Stufe, $\ell_t = \alpha\, y_t + (1-\alpha)\, \ell_{t-1}$, und sagt die Stufe voraus. `SimpleExpSmoothing` braucht eine Reihe mit Frequenz, sonst meldet statsmodels eine `ValueWarning`. Die Sonntage sind eine Reihe ohne Kalendersprünge, mit der Frequenz `"W-SUN"`.

### TODO 6 · Einfache Glättung von Hand gegen statsmodels
1. `so_gas`: `gas` an Sonntagen als Reihe mit der Frequenz `"W-SUN"`. `stufen_hand`: eine Liste mit den Stufen nach dem zweiten, dritten und vierten Sonntag, von Hand mit α = 0,5 und der Startstufe gleich dem ersten Sonntag.
2. `ses_fest`: `SimpleExpSmoothing` auf den Sonntagen des Trainingsjahrs mit `initialization_method="known"`, `initial_level` gleich dem ersten Sonntag und `fit(smoothing_level=0.5, optimized=False)`.
3. `alpha_gas`: α, von statsmodels auf den Sonntagen des Trainingsjahrs geschätzt, mit derselben festen Startstufe und `fit(method="least_squares")` wie Holt-Winters im Skript. `mae_ses`: der mittlere absolute Fehler der Prognose für den nächsten Sonntag an den Sonntagen des Prüfjahrs, mit `alpha_gas` über alle Sonntage geglättet. `mae_naiv`: derselbe Fehler für den letzten Sonntag als Prognose.

In [ ]:
# TODO 6a
so_gas = None
stufen_hand = None

# TODO 6b
ses_fest = None

# TODO 6c
alpha_gas = None
mae_ses = None
mae_naiv = None

zeige(stufen_hand, None if ses_fest is None else ses_fest.level.iloc[1:4].to_numpy(), alpha_gas, mae_ses, mae_naiv)

In [ ]:
check("TODO 6a Stufen von Hand", lambda: so_gas.index.freqstr == "W-SUN" and len(so_gas) == 104 and np.allclose(stufen_hand, [308.5, 909.25, 638.125]))
check("TODO 6b statsmodels wie von Hand", lambda: np.allclose(ses_fest.level.iloc[1:4].to_numpy(), stufen_hand) and len(ses_fest.level) == 52)
check("TODO 6c alpha und Fehler im Prüfjahr", lambda: abs(alpha_gas - 0.554) < 0.005 and abs(mae_ses - 900.5) < 1.5 and abs(mae_naiv - 870.4) < 0.1)

Von Hand und mit statsmodels kommen dieselben Stufen heraus: 308,5, 909,25 und 638,125 kWh. Das geschätzte α ist 0,554. Im Prüfjahr liegt die geglättete Prognose für den nächsten Sonntag im Mittel 900,5 kWh daneben, der letzte Sonntag nur 870,4. Beim Strom half die Glättung an Sonntagen, beim Gas nicht: Der Sonntagsverbrauch folgt dem Wetter der Woche, und die Glättung hinkt hinterher.

## 4 · Regression mit Plan, Kalender und Wetter
*Lerneinheit 4 · Skript Abschnitt 4*

Eine Regression darf nur Merkmale nutzen, die am Prognosevormittag bekannt sind: Plan, Wochentag und die Temperaturvorhersage. Gelernt wird mit der gemessenen Temperatur, prognostiziert mit der Vorhersage. Beim Gas zeigt das Skript eine **Heizgrenze**, an Arbeitstagen bei etwa 15 °C, an freien Tagen bei etwa 12 °C, und die **Gebäudemasse** lässt die Temperatur des Vortags mitzählen.

`lern` markiert die Lerntage wie im Skript. `mae_gas` passt eine lineare Regression im Trainingsjahr an und gibt den MAE auf `tage` zurück, wie `mae_test` im Skript: mit `X_lern` gelernt, mit `X_prog` prognostiziert, falls angegeben.

In [ ]:
lern = gas.notna() & ~pruef


def mae_gas(X_lern, X_prog=None):
    """Regression im Trainingsjahr auf X_lern, MAE auf den Prüftagen tage mit X_prog (ohne Angabe X_lern)."""
    m = LinearRegression().fit(X_lern[lern], gas[lern])
    p = m.predict((X_lern if X_prog is None else X_prog)[tage])
    return float((gas[tage] - p).abs().mean())


print(int(lern.sum()))

### TODO 7 · Heizgradtage und Regression für Gas
Die träge Temperatur ist $\tilde T_t = 0{,}75\, T_t + 0{,}25\, T_{t-1}$. Am ersten Tag fehlt der Vortag, dort nimmst du $T_t$ selbst, etwa mit `temp.shift(1).fillna(temp)`.
1. `gas_merkmale(temp)`: eine Funktion, die aus einer Temperaturreihe eine Tabelle mit den Spalten `eine_schicht`, `drei_schichten` (Dummies wie im Skript), `hgt_arbeit` = $\max(15 - \tilde T, 0)$ an Arbeitstagen, sonst 0, und `hgt_frei` = $\max(12 - \tilde T, 0)$ an freien Tagen, sonst 0, baut.
2. `G_mess = gas_merkmale(energie["aussentemp_c"])`. `gas_modell`: eine `LinearRegression` auf `G_mess`, angepasst an den Tagen `lern`. `mae_mess`: ihr MAE auf `tage` mit `G_mess`.
3. `einfacher`: ein Wörterbuch mit dem MAE dreier einfacherer Modelle aus `mae_gas`, alle mit der gemessenen Temperatur: „linear“ mit `arbeitstag` und der Temperatur selbst, „ohne Trägheit“ mit `arbeitstag` und den Heizgradtagen je Tagesart aus der Tagestemperatur, „mit Trägheit“ wie davor, aber aus der trägen Temperatur.

In [ ]:
# TODO 7a
gas_merkmale = None

# TODO 7b
G_mess = None
gas_modell = None
mae_mess = None

# TODO 7c
einfacher = None

zeige(None if gas_modell is None else np.r_[gas_modell.intercept_, gas_modell.coef_], mae_mess, einfacher)

In [ ]:
check("TODO 7a Merkmale", lambda: list(G_mess.columns) == ["eine_schicht", "drei_schichten", "hgt_arbeit", "hgt_frei"] and abs(G_mess.loc["2024-10-01", "hgt_arbeit"] - 2.0) < 1e-9
      and abs(G_mess["hgt_arbeit"].sum() - 2730.975) < 0.01 and abs(G_mess["hgt_frei"].sum() - 737.35) < 0.01 and float(G_mess.loc[~energie["arbeitstag"], "hgt_arbeit"].abs().sum()) == 0)
check("TODO 7b gas_modell im Trainingsjahr", lambda: gas_modell.n_features_in_ == 4 and np.allclose(np.r_[gas_modell.intercept_, gas_modell.coef_], [311.4, 470.5, 4307.8, 888.9, 600.7], atol=0.1)
      and abs(mae_mess - 549.3) < 0.1)
check("TODO 7c einfachere Modelle", lambda: np.allclose([einfacher["linear"], einfacher["ohne Trägheit"], einfacher["mit Trägheit"]], [1736.3, 806.3, 758.2], atol=0.1))

In [ ]:
# Vorgabe: Ist TODO 7 noch offen, legt diese Zelle gas_merkmale, G_mess und gas_modell an. In der Lösung ändert sie nichts.
if gas_merkmale is None:
    def gas_merkmale(temp):
        traeg = 0.75 * temp + 0.25 * temp.shift(1).fillna(temp)
        return pd.DataFrame({"eine_schicht": (energie["schichten"] == 1).astype(int), "drei_schichten": (energie["schichten"] == 3).astype(int),
                             "hgt_arbeit": (15 - traeg).clip(lower=0) * energie["arbeitstag"], "hgt_frei": (12 - traeg).clip(lower=0) * ~energie["arbeitstag"]})
if G_mess is None or gas_modell is None:
    G_mess = gas_merkmale(energie["aussentemp_c"])
    gas_modell = LinearRegression().fit(G_mess[lern], gas[lern])
print(list(G_mess.columns))

`gas_modell` hat einen Grundwert von 311,4 kWh. Eine Schicht bringt 470,5 kWh mehr, drei Schichten 4.307,8. Jeder Heizgradtag kostet an Arbeitstagen 888,9 kWh und an freien Tagen 600,7. Mit der gemessenen Temperatur liegt der MAE im Prüfjahr bei 549,3 kWh. Die einfacheren Modelle zeigen, was jeder Baustein bringt: Arbeitstag und Temperatur als gerade Linie kommen auf 1.736,3, Heizgradtage je Tagesart auf 806,3, mit träger Temperatur auf 758,2. Die Schichtzahl statt des Arbeitstags bringt den Rest bis 549,3.

### TODO 8 · Leckage messen: Vorhersage statt Messung
1. `G_vorh = gas_merkmale(energie["temp_vorhersage_c"])`: dieselben Merkmale aus der Vorhersage, auch für den Vortag.
2. `mae_vorh`: der MAE von `gas_modell` auf `tage` mit `G_vorh`. `mape_mess` und `mape_vorh`: der MAPE auf `tage` mit `G_mess` und mit `G_vorh`.

In [ ]:
# TODO 8a
G_vorh = None

# TODO 8b
mae_vorh = None
mape_mess = None
mape_vorh = None

zeige(mae_vorh, mape_mess, mape_vorh)

In [ ]:
check("TODO 8a Merkmale aus der Vorhersage", lambda: list(G_vorh.columns) == list(G_mess.columns) and abs(G_vorh.loc["2024-10-01", "hgt_arbeit"] - 2.9) < 1e-9
      and abs(G_vorh["hgt_arbeit"].sum() - 2795.95) < 0.01 and abs(G_vorh["hgt_frei"].sum() - 728.475) < 0.01 and bool((G_vorh[["eine_schicht", "drei_schichten"]] == G_mess[["eine_schicht", "drei_schichten"]]).all().all()))
check("TODO 8b ehrlicher Fehler", lambda: abs(mae_vorh - 738.5) < 0.1 and abs(mape_mess - 9.7) < 0.05 and abs(mape_vorh - 13.8) < 0.05)

In [ ]:
# Vorgabe: Ist TODO 8 noch offen, legt diese Zelle G_vorh an. In der Lösung ändert sie nichts.
if G_vorh is None:
    G_vorh = gas_merkmale(energie["temp_vorhersage_c"])
print(G_vorh.shape)

Mit der Vorhersage steigt der MAE von 549,3 auf 738,5 kWh und der MAPE von 9,7 auf 13,8 %. Mit der gemessenen Temperatur wäre die Gasprognose um ein Viertel zu gut. Beim Strom kostete die Vorhersage nur 20 kWh, weil Strom kaum an der Temperatur hängt. Je stärker eine Größe am Wetter hängt, desto teurer ist die Leckage. **738,5 ist die ehrliche Zahl.**

## 5 · Fair prüfen mit rollierendem Ursprung
*Lerneinheit 5 · Skript Abschnitt 5*

Beim Backtesting wandert der Ursprung durch das Prüfjahr. An jedem Ursprung wird mit allen Daten bis dahin gelernt (expandierendes Fenster) und genau für den Horizont des Betriebs prognostiziert. Für den Monat ab dem Ersten ist am Vormittag des Vortags alles bis zwei Tage vor dem Ersten bekannt. Monatlich neu anzupassen reicht.

### TODO 9 · Backtesting mit monatlichem Neuanpassen
1. `gas_backtest(X_lern, X_prog)`: eine Funktion wie `backtest` im Skript für Gas. An jedem Monatsersten vom 1. Oktober 2025 bis 1. September 2026 passt sie eine `LinearRegression` mit allen Tagen an, an denen Gas vorliegt und die höchstens zwei Tage vor dem Ersten liegen, und prognostiziert den Monat mit `X_prog`. Sie gibt eine Series mit allen Tagen zurück, im Trainingsjahr NaN.
2. `gas_monatlich = gas_backtest(G_mess, G_vorh)`. `mae_monatlich`: der MAE auf `tage`. `mae_je_monat`: der MAE auf `tage` je Kalendermonat als Series mit den Monaten 1 bis 12 als Index.

In [ ]:
# TODO 9a
gas_backtest = None

# TODO 9b
gas_monatlich = None
mae_monatlich = None
mae_je_monat = None

zeige(mae_monatlich, None if mae_je_monat is None else mae_je_monat.round(0).to_dict())

In [ ]:
check("TODO 9a Prognosen nur im Prüfjahr", lambda: bool(gas_monatlich[pruef].notna().all()) and bool(gas_monatlich[~pruef].isna().all()))
check("TODO 9b Fehler gesamt und je Monat", lambda: abs(mae_monatlich - 738.0) < 0.1 and list(mae_je_monat.index) == list(range(1, 13))
      and abs(mae_je_monat[8] - 280) < 1 and abs(mae_je_monat[12] - 1020) < 1 and mae_je_monat.idxmin() == 8 and mae_je_monat.idxmax() == 12)

In [ ]:
# Vorgabe: Ist TODO 9 noch offen, rechnet diese Zelle das Backtesting. In der Lösung ändert sie nichts.
if gas_monatlich is None:
    gas_monatlich = pd.Series(np.nan, index=energie.index)
    for _start in pd.date_range("2025-10-01", "2026-09-01", freq="MS"):
        _bekannt = gas.notna() & (energie.index <= _start - pd.Timedelta(days=2))
        _ziel = (energie.index >= _start) & (energie.index < _start + pd.offsets.MonthBegin(1))
        gas_monatlich[_ziel] = LinearRegression().fit(G_mess[_bekannt], gas[_bekannt]).predict(G_vorh[_ziel])
print(int(gas_monatlich.notna().sum()))

Monatlich neu angepasst liegt der MAE bei 738,0 kWh, einmal angepasst bei 738,5. Wie beim Strom ändert sich der Zusammenhang über das Jahr kaum. Je Monat reicht der MAE von 280 kWh im August bis 1.020 im Dezember. Im Winter liegt die Temperatur unter der Heizgrenze, dort zählt jeder Fehler der Vorhersage: An Arbeitstagen kostet ein Heizgradtag 889 kWh. Im Sommer liegt die Temperatur meist über der Heizgrenze, und Fehler der Vorhersage spielen kaum eine Rolle.

## 6 · Wie sicher ist die Prognose?
*Lerneinheit 6 · Skript Abschnitt 6*

Ein 90-%-Prognoseintervall aus den empirischen Quantilen der Reste: Liegen 90 % der Fehler im Training zwischen dem 5- und dem 95-%-Quantil, setzt man diese Grenzen um jede neue Prognose. Die Reste im Training rechnest du so, wie die Prognose im Betrieb entsteht, mit der Vorhersage. Ob das Intervall hält, zeigt erst die Überdeckung im Prüfjahr, getrennt für jede Gruppe, für die es gelten soll.

### TODO 10 · Intervalle und Überdeckung
1. `rest_gas_prog`: `gas` minus die Prognose von `gas_modell` mit `G_vorh`, für alle Tage. `gas_grenzen`: eine Tabelle mit dem 5- und dem 95-%-Quantil dieser Reste an den Tagen `lern`, je Tagesart: Zeilen `False` und `True` für `arbeitstag`, Spalten `0.05` und `0.95`.
2. `ueberdeckung`: ein Wörterbuch mit dem Anteil der Tage in `tage`, deren Rest zwischen den Grenzen ihrer Tagesart liegt (Grenzen eingeschlossen), für freie Tage (Schlüssel `False`), Arbeitstage (`True`) und alle (Schlüssel „alle“).
3. `grenzen_gemeinsam`: die beiden Quantile über alle Lerntage als Array. `ueberdeckung_gemeinsam`: ein Wörterbuch mit den Schlüsseln `False` und `True` wie in Schritt 2, mit diesen gemeinsamen Grenzen.

In [ ]:
# TODO 10a
rest_gas_prog = None
gas_grenzen = None

# TODO 10b
ueberdeckung = None

# TODO 10c
grenzen_gemeinsam = None
ueberdeckung_gemeinsam = None

print(None if gas_grenzen is None else gas_grenzen.round(0))
zeige(ueberdeckung, grenzen_gemeinsam, ueberdeckung_gemeinsam)

In [ ]:
check("TODO 10a Grenzen je Tagesart", lambda: np.allclose(gas_grenzen.loc[False, [0.05, 0.95]], [-825, 706], atol=1) and np.allclose(gas_grenzen.loc[True, [0.05, 0.95]], [-2250, 2245], atol=1)
      and abs(rest_gas_prog[tage].abs().mean() - 738.5) < 0.1)
check("TODO 10b Überdeckung im Prüfjahr", lambda: np.allclose([ueberdeckung[False], ueberdeckung[True], ueberdeckung["alle"]], [0.858, 0.935, 0.912], atol=0.0006))
check("TODO 10c gemeinsames Intervall", lambda: np.allclose(grenzen_gemeinsam, [-1971, 1739], atol=1) and ueberdeckung_gemeinsam[False] == 1.0 and abs(ueberdeckung_gemeinsam[True] - 0.870) < 0.0006)

In [ ]:
# Vorgabe: Ist TODO 10 noch offen, rechnet diese Zelle die Überdeckung. In der Lösung ändert sie nichts.
if ueberdeckung is None:
    _rest = gas - gas_modell.predict(G_vorh)
    _art = energie["arbeitstag"]
    _q = _rest[lern].groupby(_art[lern]).quantile([0.05, 0.95]).unstack()
    ueberdeckung = {"alle": float(((_rest >= _art.map(_q[0.05])) & (_rest <= _art.map(_q[0.95])))[tage].mean())}
print(round(ueberdeckung["alle"], 3))

An freien Tagen reicht das Intervall von −825 bis 706 kWh um die Prognose, an Arbeitstagen von −2.250 bis 2.245. Im Prüfjahr liegen 85,8 % der freien Tage und 93,5 % der Arbeitstage darin, zusammen 91,2 %. Beim Gas hält das Intervall besser als beim Strom, wo es nur 80,7 % waren. Das gemeinsame Intervall von −1.971 bis 1.739 kWh enthält jeden freien Tag und nur 87,0 % der Arbeitstage: an freien Tagen zu breit, an Arbeitstagen zu schmal.

## 7 · Wann läuft etwas aus dem Ruder?
*Lerneinheit 7 · Skript Abschnitt 7*

Überwacht wird der Prognosefehler, nicht der Rohwert. An freien Tagen streuen die Reste der Strom-Regression wenig, eine zusätzliche Last fällt dort auf. Die Shewhart-Karte schlägt an, sobald ein standardisierter Wert über 3 liegt. Die CUSUM-Karte summiert kleine Abweichungen auf, $C_t = \max(0, C_{t-1} + z_t - k)$, und schlägt an, wenn $C_t > h$; danach beginnt sie wieder bei 0. Die EWMA-Karte glättet die Werte exponentiell. Mittel und Streuung kommen aus einer ruhigen Bezugszeit, hier dem Trainingsjahr.

Die nächste Zelle rechnet die Strom-Regression aus dem Skript nach: `tabelle`, `merkmale`, `X_mess` und `modell` wie dort, und `rest` ist Strom minus die Regression mit gemessener Temperatur. Für die Überwachung ist der Tag schon abgelesen, die Messung ist hier keine Leckage.

In [ ]:
def tabelle(temp):
    t = pd.DataFrame(index=energie.index)
    t["eine_schicht"] = (energie["schichten"] == 1).astype(int)
    t["drei_schichten"] = (energie["schichten"] == 3).astype(int)
    t["soll"] = energie["soll"]
    t["hgt"] = (15 - temp).clip(lower=0)
    kgt = (temp - 18).clip(lower=0)
    t["kgt_arbeit"] = kgt * energie["arbeitstag"]
    t["kgt_frei"] = kgt * ~energie["arbeitstag"]
    return t


merkmale = ["eine_schicht", "drei_schichten", "soll", "hgt", "kgt_arbeit", "kgt_frei"]
X_mess = tabelle(energie["aussentemp_c"])
modell = LinearRegression().fit(X_mess[strom.notna() & ~pruef], strom[strom.notna() & ~pruef])
rest = strom - modell.predict(X_mess)
print(round(modell.intercept_, 1), int(rest.notna().sum()))

### TODO 11 · CUSUM auf den Resten der freien Tage
1. `frei`: True an den freien Tagen, an denen `rest` vorliegt. `SIGMA_FREI`: die Standardabweichung von `rest` an freien Tagen im Trainingsjahr. `z_frei`: `rest` an allen freien Tagen geteilt durch `SIGMA_FREI`.
2. `cusum(z, k=0.5, h=4.0)`: eine Funktion, die für eine Series `z` die CUSUM rechnet, $C$ nach jedem Alarm auf 0 setzt und die Liste der Indexwerte mit Alarm zurückgibt.
3. `alarme4 = cusum(z_frei, h=4.0)` und `alarme5 = cusum(z_frei, h=5.0)`.

In [ ]:
# TODO 11a
frei = None
SIGMA_FREI = None
z_frei = None

# TODO 11b
cusum = None

# TODO 11c
alarme4 = None
alarme5 = None

zeige(SIGMA_FREI, alarme4, alarme5)

In [ ]:
check("TODO 11a standardisierte Reste", lambda: int(frei.sum()) == 225 and abs(SIGMA_FREI - 80.4) < 0.05 and len(z_frei) == 225 and abs(z_frei.loc["2026-05-23"] - 3.60) < 0.005)
check("TODO 11b CUSUM mit Zurücksetzen", lambda: cusum(pd.Series([0.4, 1.5, 1.2, 1.8, 1.6])) == [4] and cusum(pd.Series([4.6, 0.6])) == [0] and cusum(pd.Series([4.6, 0.6]), h=5.0) == [])
check("TODO 11c Alarme an freien Tagen", lambda: len(alarme4) == 8 and [t for t in alarme4 if t <= pd.Timestamp(TRAIN_ENDE)] == [pd.Timestamp("2025-08-20")]
      and [t for t in alarme4 if t > pd.Timestamp(TRAIN_ENDE)][0] == pd.Timestamp("2026-04-19")
      and [t for t in alarme5 if t <= pd.Timestamp(TRAIN_ENDE)] == [] and alarme5[0] == pd.Timestamp("2026-04-26"))

In [ ]:
# Vorgabe: Ist TODO 11 noch offen, legt diese Zelle z_frei, cusum und alarme4 an. In der Lösung ändert sie nichts.
if z_frei is None:
    _frei = ~energie["arbeitstag"] & rest.notna()
    z_frei = rest[_frei] / rest[_frei & ~pruef].std()
if cusum is None:
    def cusum(z, k=0.5, h=4.0):
        c, alarme = 0.0, []
        for zeit, wert in z.items():
            c = max(0.0, c + wert - k)
            if c > h:
                alarme.append(zeit)
                c = 0.0
        return alarme
if alarme4 is None:
    alarme4 = cusum(z_frei, h=4.0)
print(len(z_frei), len(alarme4))

In [ ]:
c_verlauf, c_ = [], 0.0
for wert in z_frei:
    c_ = max(0.0, c_ + wert - 0.5)
    c_verlauf.append(c_)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 4.6), sharex=True)
ax1.scatter(z_frei.index, z_frei, s=8, color="tab:gray")
ax1.axhline(3, color="tab:red", lw=1, label="Shewhart-Grenze 3")
ax1.set_ylabel("z an freien Tagen")
ax1.legend(loc="upper left")
ax2.plot(z_frei.index, c_verlauf, color="tab:blue", lw=1, label="CUSUM ohne Zurücksetzen")
ax2.axhline(4, color="tab:red", lw=1, label="h = 4")
for t in alarme4:
    ax2.axvline(t, color="tab:orange", lw=0.6)
ax2.set_ylabel("C")
ax2.legend(loc="upper left")
ax1.set_title("Reste an freien Tagen: Die CUSUM steigt im Frühjahr 2026 lange vor der Shewhart-Grenze")
plt.show()

Die Reste an freien Tagen streuen im Trainingsjahr mit 80,4 kWh. Mit $h = 4$ schlägt die CUSUM im Prüfjahr zum ersten Mal am 19. April 2026 an. Im Trainingsjahr gibt es einen Fehlalarm am 20. August 2025, in der Werksruhe. Mit $h = 5$ kommt der erste Alarm eine Woche später, am 26. April 2026, und im Trainingsjahr keiner. Die größere Grenze kostet eine Woche und spart den Fehlalarm.

### TODO 12 · EWMA und Shewhart im Vergleich
1. `ewma(z, lam=0.2, L=3.0)`: eine Funktion, die $E_t = \lambda z_t + (1-\lambda) E_{t-1}$ mit $E_0 = 0$ rechnet und die Liste der Indexwerte zurückgibt, an denen $E_t$ über $L\sqrt{\lambda/(2-\lambda)}$ liegt.
2. `alarme_ewma = ewma(z_frei)`. `alarme_shewhart`: die Liste der Indexwerte von `z_frei` mit einem Wert über 3.
3. `abstand_tage`: um wie viele Tage der erste Alarm der Shewhart-Karte im Prüfjahr nach dem ersten Alarm von `alarme4` im Prüfjahr kommt, als ganze Zahl.

In [ ]:
# TODO 12a
ewma = None

# TODO 12b
alarme_ewma = None
alarme_shewhart = None

# TODO 12c
abstand_tage = None

zeige(None if alarme_ewma is None else alarme_ewma[:3], None if alarme_shewhart is None else alarme_shewhart[:3], abstand_tage)

In [ ]:
check("TODO 12a EWMA an kurzen Reihen", lambda: ewma(pd.Series([3.0, 3.0, 0.0])) == [1] and ewma(pd.Series([3.0, 3.0, 0.0]), L=4.0) == [])
check("TODO 12b erste Alarme", lambda: alarme_ewma[0] == pd.Timestamp("2026-04-26") and alarme_shewhart[0] == pd.Timestamp("2026-05-23") and len(alarme_shewhart) == 7)
check("TODO 12c Abstand in Tagen", lambda: abstand_tage == 34)

In [ ]:
# Vorgabe: Ist TODO 12 noch offen, legt diese Zelle ewma und die Alarme an. In der Lösung ändert sie nichts.
if ewma is None:
    def ewma(z, lam=0.2, L=3.0):
        e, alarme = 0.0, []
        for zeit, wert in z.items():
            e = lam * wert + (1 - lam) * e
            if e > L * np.sqrt(lam / (2 - lam)):
                alarme.append(zeit)
        return alarme
if alarme_ewma is None:
    alarme_ewma = ewma(z_frei)
if alarme_shewhart is None:
    alarme_shewhart = list(z_frei.index[z_frei > 3])
print(len(alarme_ewma), len(alarme_shewhart))

Die EWMA schlägt am 26. April 2026 zum ersten Mal an, die Shewhart-Karte erst am 23. Mai, 34 Tage nach der CUSUM mit $h = 4$. Im Trainingsjahr melden beide nichts. Eine Verschiebung um gut eine Standardabweichung finden CUSUM und EWMA Wochen früher als die Shewhart-Karte, wie die Lauflängen im Skript erwarten lassen.

### TODO 13 · Ergebnisbericht als Tabelle
1. `bericht`: eine Tabelle mit den Zeilen „saisonal naiv“, „Regression mit Vorhersage“ und „monatlich neu angepasst“ und den Spalten MAE, RMSE und MASE auf `tage`, mit `fehlermasse` und `MASE_NENNER_GAS`. Die Prognosen stehen in `gas_prognosen`, in `gas_modell` mit `G_vorh` und in `gas_monatlich`.
2. `erste_alarme`: ein Wörterbuch mit dem ersten Alarm im Prüfjahr als Text im Format TT.MM.JJJJ für „CUSUM h=4“, „EWMA“ und „Shewhart“.
3. `fazit`: drei Sätze für das Energiemanagement als Text: wie gut die Gasprognose ist, wie verlässlich das Intervall ist (`ueberdeckung["alle"]`) und was die Überwachung meldet.

In [ ]:
# TODO 13a
bericht = None

# TODO 13b
erste_alarme = None

# TODO 13c
fazit = None

print(None if bericht is None else bericht.round(3))
zeige(erste_alarme)
print(fazit)

In [ ]:
check("TODO 13a Bericht", lambda: list(bericht.index) == ["saisonal naiv", "Regression mit Vorhersage", "monatlich neu angepasst"] and list(bericht.columns) == ["MAE", "RMSE", "MASE"]
      and np.allclose(bericht["MASE"], [0.904, 0.258, 0.258], atol=0.0006) and np.allclose(bericht["MAE"], [2583.2, 738.5, 738.0], atol=0.1))
check("TODO 13b erste Alarme", lambda: erste_alarme == {"CUSUM h=4": "19.04.2026", "EWMA": "26.04.2026", "Shewhart": "23.05.2026"})
check("TODO 13c Fazit in drei Sätzen", lambda: isinstance(fazit, str) and len(fazit.split(". ")) >= 3 and len(fazit) >= 150)

Für das Energiemanagement: Die Gasprognose mit Plan und Heizgradtagen hat einen MASE von 0,258, die saisonal naive Prognose 0,904. Monatlich neu anzupassen ändert nichts Wesentliches. Das Intervall je Tagesart hält im Prüfjahr mit 91,2 % fast die nominalen 90 %. Bei der Überwachung meldet die CUSUM mit $h = 4$ die Verschiebung an den freien Tagen am 19. April, die EWMA am 26. April und die Shewhart-Karte erst am 23. Mai 2026. Ein Alarm ist ein Anlass für einen Rundgang, kein Befund.

Die beiden Bonusaufgaben gehen dem Horizont beim Gas nach und überwachen eine einzelne Maschine statt einer ganzen Linie.

### Bonus 1 · Rest-Lag und Horizont beim Gas
1. `rest_gas`: `gas` minus die Prognose von `gas_modell` mit `G_mess`, für alle Tage.
2. `mae_lag`: ein Wörterbuch mit den Schlüsseln 1 und 2 und dem MAE aus `mae_gas`, wenn du den Rest von vor 1 oder 2 Tagen (fehlende Werte als 0) als Spalte `rest` an `G_mess` und an `G_vorh` hängst: gelernt mit `G_mess`, prognostiziert mit `G_vorh`.
3. `ak_rest_gas`: die Autokorrelation bei Lag 1 von `rest_gas` im Trainingsjahr, nur an den Tagen `lern` (sonst NaN), mit `autocorr`.

In [ ]:
# Bonus 1
rest_gas = None
mae_lag = None
ak_rest_gas = None

zeige(mae_lag, ak_rest_gas)

In [ ]:
check("Bonus 1 Rest-Lag", lambda: np.allclose([mae_lag[1], mae_lag[2]], [738.4, 739.9], atol=0.1) and abs(ak_rest_gas - 0.006) < 0.0006)

Mit dem Rest von gestern sinkt der MAE nur von 738,5 auf 738,4 kWh, mit dem Rest von vorgestern steigt er auf 739,9. Die Reste der Gas-Regression hängen im Training von Tag zu Tag kaum zusammen, die Autokorrelation bei Lag 1 ist 0,006. Die träge Temperatur hat den Zusammenhang schon aufgenommen, den beim Strom der Rest von gestern trug.

### Bonus 2 · Sensortausch an einer einzelnen Maschine
1. `zs_m2`: der Logarithmus der Schwingung von M-02 je Datum und Schicht (Schichten in der Reihenfolge Früh, Spät, Nacht, fehlende Werte gestrichen), standardisiert mit Mittel und Standardabweichung von September bis Dezember 2025, für Januar bis März 2026.
2. `sprung_m2`: das Mittel von `zs_m2` ab dem 2. März 2026 minus das Mittel davor.
3. `erste_m2`: ein Wörterbuch mit dem ersten Alarm als Paar (Datum, Schicht) für „Shewhart“, „Shewhart nach Tausch“ (erster Wert über 3 ab dem 2. März), „CUSUM“ mit `cusum(zs_m2, h=5.0)` und „EWMA“ mit `ewma(zs_m2)`.

In [ ]:
# Bonus 2
zs_m2 = None
sprung_m2 = None
erste_m2 = None

zeige(sprung_m2, erste_m2)

In [ ]:
check("Bonus 2 einzelne Maschine", lambda: abs(sprung_m2 - 2.35) < 0.005 and erste_m2["Shewhart"] == ("2026-02-16", "Spät") and erste_m2["Shewhart nach Tausch"] == ("2026-03-04", "Früh")
      and erste_m2["CUSUM"] == ("2026-03-02", "Nacht") and erste_m2["EWMA"] == ("2026-03-02", "Nacht"))

An M-02 allein springt die Schwingung um 2,35 Standardabweichungen, mehr als im Median der Linie. Trotzdem ist die einzelne Maschine schwerer zu überwachen: Die Shewhart-Karte schlägt schon am 16. Februar 2026 in der Spätschicht an, ohne Tausch, und nach dem Tausch erst am 4. März in der Frühschicht. CUSUM und EWMA melden den Tausch in der Nachtschicht am 2. März, der dritten Schicht danach. Der Median über fünf Maschinen dämpft das Rauschen einzelner Schichten, deshalb meldeten im Skript alle Karten schon in der Spätschicht.

## Zum Nachdenken

1. In TODO 5 liegt der MAPE der saisonal naiven Prognose bei 87,5 %, der MASE bei 0,904. Welche Zahl gibst du dem Energiemanagement, und warum?
2. In TODO 8 kostet die Vorhersage beim Gas ein Viertel mehr Fehler, beim Strom fast nichts. Was heißt das für die Wahl der Merkmale und für die Bewertung?
3. Das Intervall in TODO 10 hält beim Gas mit 91,2 %. Reicht das, um es im Winter für den Einkauf einer Woche zu nutzen?
4. Die CUSUM mit $h = 4$ meldet einen Fehlalarm in der Werksruhe. Was würdest du ändern: die Grenze, die Bezugszeit oder die Tage, die überwacht werden?
5. Im Juli fallen die Reste an freien Tagen zurück. Was müsstest du wissen, um zu sagen, dass eine Maßnahme gewirkt hat?

<details><summary>Hinweise zu den Antworten</summary>

1. Den MAE und den MASE. Der MAPE teilt durch den tatsächlichen Verbrauch, und an freien Sommertagen sind das wenige hundert kWh. Dort entstehen MAPE-Werte von mehreren hundert Prozent, obwohl die Fehler in kWh klein sind. Für Einkauf und Spitzenlast zählen kWh.
2. Je stärker eine Größe am Wetter hängt, desto mehr kostet die Unsicherheit der Vorhersage. Bewertet wird immer mit den Merkmalen, die am Prognosevormittag bekannt sind. Gelernt werden darf mit der Messung, weil sie den Zusammenhang am genauesten beschreibt.
3. Nicht ohne Weiteres. Die Überdeckung gilt für einzelne Tage über das ganze Jahr. Im Winter sind die Fehler größer, und für eine Wochensumme zählt, ob die Fehler aufeinanderfolgender Tage zusammenhängen. Prüfe die Überdeckung für die Gruppe, für die das Intervall gelten soll.
4. Die Werksruhe ist eine eigene Lage: lange Folgen freier Tage, andere Lasten. Man könnte sie aus der Bezugszeit nehmen oder getrennt überwachen. Eine größere Grenze spart den Fehlalarm, meldet aber eine echte Verschiebung eine Woche später. Die Wahl gehört begründet, bevor überwacht wird, nicht nach einem Blick auf das Prüfjahr.
5. Was wann geändert wurde, und ob sich in derselben Zeit Wetter, Plan oder andere Lasten geändert haben. Ein Vorher-nachher-Vergleich ist noch kein Nachweis.
</details>

## Wie es weitergeht

Du hast Ausfälle mit Nachbuchung bereinigt, den Plan angehängt, das Gas zerlegt, Autokorrelation und Glättung von Hand gerechnet, Messlatten mit dem richtigen Horizont gebaut und eine Regression mit Heizgradtagen ohne Leckage. Mit rollierendem Ursprung und Intervallen hast du sie fair geprüft und die Reste an freien Tagen mit drei Karten überwacht. Offen bleibt, ob das Zurückfallen der Reste im Juli an einer Maßnahme liegt. **Modul 12 „Experimente und A/B-Tests“** fragt: Wie zeige ich, dass eine Maßnahme wirkt?